# Lab 12 - Bug-Fixer Agent That Opens a Pull Request

**Section E · Orchestration, Integration & Production**  ·  **Chapter 14 · Files & GitHub Workflows**  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

> **Udemy Labs note:** the setup cell below configures `ANTHROPIC_API_KEY`, the GitHub repo token, the repo URL, and your GitHub Managed Agents vault id for this kernel session. The repo token must be a fine-grained PAT scoped to the one repo you target; the vault authenticates the GitHub MCP server with a bearer-token credential backed by your fine-grained PAT.

## Goal
Build a bug-fixer agent that runs the full clone-edit-branch-PR loop in a single session. You attach a GitHub repository as a session resource with a fine-grained token, the agent reproduces a planted bug, edits the minimum code to fix it, creates a new branch, and opens a pull request. You then retrieve the PR URL from the run.

The repo attaches as a first-class resource: the token clones the repo once and wires into the local git remote. The token never reaches the sandbox where Claude's code runs. To push and open a PR, you pair the repo with the GitHub MCP server authenticated through a Claude Managed Agents vault.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- A small Python repo **you own** with a planted bug and a failing test. Fast path: open the course template at [github.com/puria-izady/managed-agents-bugfixer-sample](https://github.com/puria-izady/managed-agents-bugfixer-sample) and click **Use this template** to create your own copy (an off-by-one in a `factorial` helper with a failing test in `tests/test_math.py`). You need a repo you own so the fine-grained token can push a branch and open a PR. Use the URL of *your* copy as `GITHUB_REPO_URL` below.
- A **fine-grained** GitHub personal access token in the `GITHUB_TOKEN` env var. When creating it in GitHub, choose **Fine-grained token**, set **Repository access** to **Only select repositories**, select your lab repo, then grant these repository permissions: `Contents: Read and write` and `Pull requests: Read and write`. This token is only for the mounted repo resource.
- A GitHub MCP connection configured in Claude Managed Agents Vaults. In Claude Console, create the GitHub MCP credential with **Bearer Token** auth and paste the same fine-grained PAT as the bearer token. Paste the resulting vault id into `GITHUB_VAULT_ID`. Leave `GITHUB_MCP_URL` blank unless the vault contains multiple MCP credentials and the code cannot infer the GitHub one.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From labs/code, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

if not os.environ.get("GITHUB_TOKEN"):
    os.environ["GITHUB_TOKEN"] = getpass.getpass(
        "GitHub fine-grained repo token for the mounted repo: "
    ).strip()

current_repo_url = os.environ.get("GITHUB_REPO_URL", "").strip()
if current_repo_url:
    repo_url = input(
        f"GitHub repo URL [{current_repo_url}] (press Enter to keep): "
    ).strip()
    if repo_url:
        os.environ["GITHUB_REPO_URL"] = repo_url
else:
    os.environ["GITHUB_REPO_URL"] = input(
        "GitHub repo URL (https://github.com/owner/repo): "
    ).strip()

print("Configure your GitHub MCP Managed Agents vault for this notebook. Use a bearer-token credential backed by your fine-grained PAT.")
existing_vault_id = os.environ.get("GITHUB_VAULT_ID", "").strip()
if existing_vault_id.startswith("sk-ant-") or (existing_vault_id and not existing_vault_id.startswith("vlt_")):
    print("Clearing invalid GITHUB_VAULT_ID. It must be a vault id starting with 'vlt_', not an API key.")
    os.environ.pop("GITHUB_VAULT_ID", None)

if not os.environ.get("GITHUB_VAULT_ID"):
    os.environ["GITHUB_VAULT_ID"] = input(
        "GitHub MCP vault ID from Claude Console (vlt_...): "
    ).strip()

# Usually optional: the notebook can read the MCP URL from the vault credential.
if not os.environ.get("GITHUB_MCP_URL"):
    mcp_url = input("GitHub MCP URL (optional; press Enter to read it from the vault): ").strip()
    if mcp_url:
        os.environ["GITHUB_MCP_URL"] = mcp_url

print("ANTHROPIC_API_KEY configured for this notebook session.")
print("GITHUB_REPO_URL configured:", os.environ["GITHUB_REPO_URL"])
print("GITHUB_VAULT_ID configured for this notebook session:", os.environ["GITHUB_VAULT_ID"])

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

from anthropic import Anthropic

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."
assert os.environ.get("GITHUB_TOKEN"), "Set GITHUB_TOKEN (fine-grained repo PAT) first."
assert os.environ.get("GITHUB_REPO_URL"), "Set GITHUB_REPO_URL first."
assert os.environ.get("GITHUB_VAULT_ID"), "Set GITHUB_VAULT_ID first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
REQUIRED_REPO_FILES = ["tests/test_math.py", "src/math.py"]
client = Anthropic()


def validate_vault_id(vault_id: str) -> None:
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError("Set GITHUB_VAULT_ID to your Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            "GITHUB_VAULT_ID currently contains an Anthropic API key. Paste the "
            "GitHub vault id from Claude Console instead; it should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"GITHUB_VAULT_ID should start with 'vlt_'. Got: {vault_id!r}")


def validate_mcp_url(url: str) -> None:
    """Catch missing or placeholder MCP URLs before agent creation."""
    parsed = urlparse(url)
    if not url or "REPLACE-ME" in url or parsed.scheme != "https" or not parsed.netloc:
        raise RuntimeError(
            "Set GITHUB_MCP_URL to a valid https MCP endpoint, or use a "
            "GITHUB_VAULT_ID whose credential contains an MCP server URL."
        )


def mcp_url_from_vault(vault_id: str) -> str:
    """Read the GitHub MCP URL from an existing vault credential."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(getattr(credential, "auth", None), "type", None) == "mcp_oauth"
    ]
    github_credentials = [
        credential for credential in mcp_credentials
        if "github" in (
            f"{getattr(credential, 'display_name', '') or ''} "
            f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '')}"
        ).lower()
    ]

    if len(github_credentials) == 1:
        return github_credentials[0].auth.mcp_server_url
    if len(mcp_credentials) == 1:
        return mcp_credentials[0].auth.mcp_server_url

    names = [
        f"{getattr(credential, 'display_name', '') or credential.id}: "
        f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '<no mcp url>')}"
        for credential in mcp_credentials
    ]
    raise RuntimeError(
        "Could not uniquely identify the GitHub MCP credential in "
        f"vault {vault_id}. Set GITHUB_MCP_URL explicitly. "
        f"Found MCP credentials: {names or 'none'}"
    )


def resolve_github_connection() -> tuple[str, str]:
    """Return (vault_id, mcp_url), deriving the URL from the vault when possible."""
    vault_id = os.environ.get("GITHUB_VAULT_ID", "").strip()
    validate_vault_id(vault_id)

    mcp_url = os.environ.get("GITHUB_MCP_URL", "").strip() or mcp_url_from_vault(vault_id)
    validate_mcp_url(mcp_url)
    os.environ["GITHUB_MCP_URL"] = mcp_url
    print("github vault.id =", vault_id, "(existing vault)")
    print("github MCP URL =", mcp_url)
    return vault_id, mcp_url


def parse_github_repo(repo_url: str) -> tuple[str, str]:
    """Return (owner, repo) from a GitHub HTTPS URL."""
    parsed = urlparse(repo_url)
    path_parts = [part for part in parsed.path.strip("/").split("/") if part]
    if parsed.netloc != "github.com" or len(path_parts) < 2:
        raise RuntimeError(
            "GITHUB_REPO_URL must look like https://github.com/owner/repo. "
            f"Got: {repo_url!r}"
        )
    owner, repo = path_parts[:2]
    return owner, repo.removesuffix(".git")


def github_api_get(path: str, token: str):
    """Small GitHub API helper for local preflight checks."""
    request = Request(
        f"https://api.github.com{path}",
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urlopen(request, timeout=20) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        hint = (
            "Check that GITHUB_REPO_URL points to your copy of the repo and "
            "GITHUB_TOKEN is a fine-grained PAT with Contents read/write on that repo."
        )
        if exc.code == 404 and path.startswith("/repos/") and "/contents" not in path:
            hint = (
                "GitHub returns 404 when the repo does not exist, and also when "
                "the repo is private but this token is not authorized for it. "
                "Open GITHUB_REPO_URL in your browser while signed in, then edit "
                "the fine-grained PAT so Repository access includes exactly this repo."
            )
        elif exc.code == 404 and "/contents" in path:
            hint = (
                "The repo is visible to the token, but the expected lab file was "
                "not found on the selected branch. Confirm you used the course "
                "template repo or pushed the template_repo files, and check GITHUB_BRANCH."
            )
        raise RuntimeError(
            f"GitHub API preflight failed for {path}: HTTP {exc.code}. "
            f"{hint} "
            f"Details: {detail[:500]}"
        ) from exc


def preflight_github_repo(repo_url: str, token: str) -> tuple[str, str, str]:
    """Verify the repo token can read the target repo and expected lab files."""
    owner, repo = parse_github_repo(repo_url)
    repo_info = github_api_get(f"/repos/{owner}/{repo}", token)
    default_branch = os.environ.get("GITHUB_BRANCH", "").strip() or repo_info.get("default_branch", "main")

    root = github_api_get(f"/repos/{owner}/{repo}/contents?ref={default_branch}", token)
    if not isinstance(root, list) or not root:
        raise RuntimeError(
            f"GitHub repo {owner}/{repo} is empty on branch {default_branch!r}. "
            "Use the course template to create a populated repo, or push the "
            "template_repo files before running this lab."
        )

    root_names = ", ".join(sorted(item.get("name", "") for item in root if item.get("name")))
    print(f"GitHub preflight: {owner}/{repo}@{default_branch} root contains: {root_names}")

    for file_path in REQUIRED_REPO_FILES:
        github_api_get(f"/repos/{owner}/{repo}/contents/{file_path}?ref={default_branch}", token)
    print("GitHub preflight: required lab files are present.")
    return owner, repo, default_branch


GITHUB_VAULT_ID, GITHUB_MCP_URL = resolve_github_connection()
GITHUB_OWNER, GITHUB_REPO, GITHUB_BRANCH = preflight_github_repo(
    os.environ["GITHUB_REPO_URL"], os.environ["GITHUB_TOKEN"]
)
print(
    "GitHub MCP reminder: the vault credential should be a bearer-token credential backed by your fine-grained PAT. "
    "If MCP repo or PR calls return 404, reconnect/reauthorize the GitHub "
    "credential in the vault so it can access this same repo."
)
print("SDK ready, model:", MODEL)

### Step 1 - Create the bug-fixer agent
The agent gets two toolsets: the built-in **agent toolset** for local file ops (read, edit, write, bash) and an **MCP toolset** for the GitHub operations (branch, commit, push, open PR). The MCP server is declared on the agent, while the GitHub credential itself is supplied later through `vault_ids` on the session.

In [ ]:
agent = client.beta.agents.create(
    name="Bug Fixer",
    model=MODEL,
    system=(
        "You are a bug-fixing agent. Steps:\n"
        "1) Read the repo at /workspace/repo.\n"
        "2) Run the tests to reproduce the failure.\n"
        "3) Fix the failing test by editing only the minimum code.\n"
        "4) Re-run tests to confirm green.\n"
        "5) Use the GitHub MCP against exactly this repository: "
        f"owner `{GITHUB_OWNER}`, repo `{GITHUB_REPO}`, base branch `{GITHUB_BRANCH}`. "
        "Never use any other owner, repo, or base branch.\n"
        "6) Use the GitHub MCP to: create a branch named "
        "`claude-fix-<short-issue>`, commit the change, push, and open "
        "a PR with a clear description of what changed and why. "
        "Do not run `git push` against the local origin; the mounted repo "
        "remote is an internal resource URL. If any GitHub MCP call returns "
        "404 for this owner/repo/base branch, stop and report that the vault "
        "GitHub credential cannot access the repo or branch; do not try "
        "`git push`. Report the PR URL when you are done."
    ),
    # Pair the mounted repo with the GitHub MCP so the agent can push and open a PR.
    mcp_servers=[{"type": "url", "name": "github", "url": GITHUB_MCP_URL}],
    tools=[
        {"type": "agent_toolset_20260401"},
        {"type": "mcp_toolset", "mcp_server_name": "github"},
    ],
    betas=BETAS,
)
print("agent.id =", agent.id)

### Step 2 - Create a cloud environment
A cloud environment with `pytest` installed and **limited** networking to GitHub and the MCP server. `allow_mcp_servers` lets the GitHub MCP reach out; `allow_package_managers` lets the sandbox install the test deps.

In [ ]:
env = client.beta.environments.create(
    name="codefix-env",
    config={
        "type": "cloud",
        "packages": {"pip": ["pytest"]},
        "networking": {
            "type": "limited",
            "allowed_hosts": [
                "api.github.com",
                "github.com",
            ],
            "allow_mcp_servers": True,
            "allow_package_managers": True,
        },
    },
    betas=BETAS,
)
print("env.id =", env.id)

### Step 3 - Attach the GitHub repo as a session resource
Create the session with a `github_repository` resource. The `authorization_token` clones the repo and configures the git remote; it stays outside the sandbox where Claude's code runs. The GitHub MCP credential comes from `GITHUB_VAULT_ID` via `vault_ids`; in Claude Console we create it as Bearer Token auth backed by the fine-grained PAT, so the token is not exposed to the agent runtime. Mounting the repo as one clone (rather than uploading files one by one) also keeps you well under the 100-file session cap.

In [ ]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    resources=[{
        "type": "github_repository",
        "url": os.environ["GITHUB_REPO_URL"],
        "mount_path": "/workspace/repo",
        "checkout": {"type": "branch", "name": GITHUB_BRANCH},
        "authorization_token": os.environ["GITHUB_TOKEN"],
    }],
    vault_ids=[GITHUB_VAULT_ID],
    title="Fix failing test",
    betas=BETAS,
)
print("session.id =", session.id)

### Step 4 - Run the fix loop and capture the PR URL
Send one user message describing the failing test. The agent runs the tests to reproduce, edits the minimum code, re-runs to confirm green, then uses the MCP to create a branch, commit, push, and open the PR.

The open-PR MCP call returns the new pull request URL in its result. We scan `agent.mcp_tool_result` events for the first GitHub PR link and print it at the end.

In [ ]:
def _extract_pr_url(event):
    """Best-effort scan of an MCP tool result for a pull request URL.

    MCP result shapes vary by server, so we walk the content blocks and
    return the first string that looks like a GitHub PR link.
    """
    content = getattr(event, "content", None) or []
    for block in content:
        text = getattr(block, "text", None)
        if isinstance(text, str) and "github.com" in text and "/pull/" in text:
            for token in text.split():
                if "github.com" in token and "/pull/" in token:
                    return token.strip().strip(".,)")
    return None


pr_url = None

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "There's a failing test in tests/test_math.py. First run "
                "`pwd && ls -la /workspace /workspace/repo && git -C "
                "/workspace/repo status --short && find /workspace/repo "
                "-maxdepth 3 -type f | sort | head -100` to verify the "
                "repo mounted. If /workspace/repo has no files, stop and "
                "say the GitHub repository resource did not mount. "
                f"Otherwise find the failure, fix it, and open a PR with the fix. "
                f"Use the GitHub MCP with owner `{GITHUB_OWNER}`, repo `{GITHUB_REPO}`, "
                f"and base branch `{GITHUB_BRANCH}`. Do not use `git push`. "
                "If the GitHub MCP returns 404 for that repo or branch, stop "
                "and explain that the vault GitHub credential needs repo access."
            ),
        }],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.tool_use":
            print(f"\n[tool: {event.name}]")
        elif event.type == "agent.mcp_tool_use":
            print(f"\n[mcp: {event.name}]")
        elif event.type == "agent.mcp_tool_result":
            pr_url = _extract_pr_url(event) or pr_url
        elif event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

if pr_url:
    print(f"\nPR: {pr_url}")
else:
    print(
        "\nNo PR URL captured from MCP results. "
        "Check the agent's closing message for the link."
    )

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
A new branch on the repo (for example `claude-fix-math`) and an open pull request carrying the fix with a clean, minimal diff and a clear description. The stream shows the loop:

```
session.id = sesn_01...

[tool: bash] pytest -> 1 failing
[tool: read] tests/test_math.py
[tool: read] src/math.py
[tool: edit] fix the off-by-one
[tool: bash] pytest -> all green
[mcp: github_create_branch] claude-fix-math
[mcp: github_commit_file] src/math.py
[mcp: github_open_pr] Fix off-by-one in factorial
--- session idle ---

PR: https://github.com/your-org/your-repo/pull/42
```

## Troubleshooting
- **Fine-grained token scopes.** Use a fine-grained personal access token, not a classic one. In GitHub's token form, set **Repository access** to **Only select repositories**, select the single lab repo, then grant exactly two repository permissions: `Contents: Read and write` (to push the branch) and `Pull requests: Read and write` (to open the PR). This token is for the `github_repository` resource, not the MCP credential.
- **Repo caching.** Repos are cached across sessions, so a second run on the same repo starts faster. If you pushed changes outside the agent and the session still sees the old tree, start a fresh session to refresh the cache rather than expecting a mid-session re-clone.
- **The 100-file cap.** A session can hold up to 100 mounted files. A small repo is well under that, but if you mount a large repo as individual files you can hit it. Mount the repo as a `github_repository` resource (one clone) rather than uploading files one by one, and for very large trees mount a single archive and extract it in-session.
- **GitHub MCP auth.** If you see `session.error: mcp_auth_failed`, the MCP server could not authenticate. Confirm `GITHUB_VAULT_ID` points to the vault with the GitHub MCP bearer-token credential and that the credential URL matches the agent's `GITHUB_MCP_URL`.
- **MCP `GET /repos/...` returns 404.** This means the repo token preflight can see the repo, but the GitHub bearer-token credential inside the vault cannot. Update the GitHub MCP credential in Claude Console to use Bearer Token auth with the fine-grained PAT for this repo.
- **Could not uniquely identify the GitHub MCP credential.** The vault has multiple MCP credentials. Set `GITHUB_MCP_URL` explicitly to the GitHub MCP endpoint stored in that vault.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste these prompts in order.

1. Scaffold the script:

> Write a Python script `lab12.py` using the Anthropic SDK `client.beta.*` Agents APIs. It creates an agent named "Bug Fixer" on model `claude-haiku-4-5-20251001` with tools `agent_toolset_20260401` and an `mcp_toolset` for a GitHub MCP server. It creates a cloud environment with pytest installed and limited networking to GitHub and the MCP server. It uses my existing GitHub Managed Agents vault from `GITHUB_VAULT_ID`, reads the GitHub MCP URL from that vault when `GITHUB_MCP_URL` is unset, and attaches my repo as a `github_repository` resource mounted at `/workspace/repo` using a fine-grained token from `GITHUB_TOKEN`.

2. Wire the fix loop:

> In the same script, send one user message: "There's a failing test in tests/test_math.py. Find it, fix it, and open a PR with the fix." Stream the session events and print each tool use, MCP tool use, and assistant text. Stop on session idle. After the run, print the pull request URL returned by the open-PR MCP call.

3. Run and report:

> Run `python lab12.py` with my env vars set. Show me the streamed tool calls and the final PR URL. If the PR did not open, read the error and tell me whether the repo token scope or the GitHub vault/MCP auth step is missing.